carga_de_Librerias

librerias para manejar datos y numeros

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from io import StringIO

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Input

import copernicusmarine
import xarray as xr
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import tensorflow as tf


C:\Users\Jesus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CARGAR LOS DATOS

In [2]:
archivo = "open-meteo-9.71N84.87W0m.csv"

with open(archivo, "r", encoding="utf-8") as f:
    lineas = f.readlines()

bloques = []
bloque_actual = []
for linea in lineas:
    if linea.strip() == "":
        if bloque_actual:
            bloques.append(bloque_actual)
            bloque_actual = []
    else:
        bloque_actual.append(linea)
if bloque_actual:
    bloques.append(bloque_actual)


bloque_oleaje = max(bloques, key=len)
df = pd.read_csv(StringIO("".join(bloque_oleaje)))


df.columns = [c.split(" (")[0] for c in df.columns]
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")
df = df.sort_index()

print("Datos cargados:", df.shape)
print(df.head())

Datos cargados: (30696, 3)
                     wave_height  wave_period  wave_direction
time                                                         
2023-01-01 00:00:00         1.08        11.95             193
2023-01-01 01:00:00         1.10        12.10             193
2023-01-01 02:00:00         1.12        12.25             192
2023-01-01 03:00:00         1.14        12.35             192
2023-01-01 04:00:00         1.16        12.40             191


In [3]:
fecha_inicio = df.index.min().strftime("%Y-%m-%dT%H:%M:%S")
fecha_fin = df.index.max().strftime("%Y-%m-%dT%H:%M:%S")

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao"],
    minimum_longitude=-85.1,
    maximum_longitude=-84.6,
    minimum_latitude=9.5,
    maximum_latitude=10,
    start_datetime=fecha_inicio,
    end_datetime=fecha_fin,
    minimum_depth=0.49402499198913574,
    maximum_depth=0.49402499198913574,
    output_filename="sst_copernicus.nc",
    output_directory=".",
)



ds = xr.open_dataset("sst_copernicus.nc")
sst = ds["thetao"].mean(dim=["latitude", "longitude"])
if "depth" in sst.dims:
    sst = sst.isel(depth=0)

df_sst = sst.to_dataframe(name="sst").reset_index()[["time", "sst"]]
df_sst["time"] = pd.to_datetime(df_sst["time"])
df_sst = df_sst.set_index("time").sort_index()
df_sst = df_sst.resample("1H").interpolate("linear")


df = df.join(df_sst, how="left")

INFO - 2026-07-10T17:46:06Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:Copernicus Marine password:

INFO - 2026-07-10T17:46:58Z - Selected dataset version: "202311"
INFO - 2026-07-10T17:46:58Z - Selected dataset part: "default"
WARNING - 2026-07-10T17:46:58Z - Some of your subset selection [2023-01-01 00:00:00+00:00, 2026-07-02 23:00:00+00:00] for the time dimension exceed the dataset coordinates [1993-01-01 00:00:00+00:00, 2026-05-26 00:00:00+00:00]
INFO - 2026-07-10T17:47:03Z - Total size of the download: 115.25 KB.
C:\Users\Jesus\AppData\Local\Temp\ipykernel_10312\621739517.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_sst = df_sst.resample("1H").interpolate("linear")


limpiar nombres de columnas (quitar las unidades entre parentesis)

In [4]:
df.columns = [c.split(" (")[0] for c in df.columns]

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time")

df = df.sort_index()

agregar la temperatura superficial del mar (SST) como variable extra, usando la API de Copernicus (Dataset 2 del proyecto). Requiere haber hecho una vez: copernicusmarine login

In [5]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 9.708336,
    "longitude": -84.87499,
    "start_date": df.index.min().strftime("%Y-%m-%d"),
    "end_date": df.index.max().strftime("%Y-%m-%d"),
    "hourly": "wind_speed_10m,surface_pressure",
    "timezone": "America/Costa_Rica",
}

respuesta = requests.get(url, params=params)
datos_clima = respuesta.json()

df_clima = pd.DataFrame({
    "time": pd.to_datetime(datos_clima["hourly"]["time"]),
    "wind_speed": datos_clima["hourly"]["wind_speed_10m"],
    "pressure": datos_clima["hourly"]["surface_pressure"],
}).set_index("time")

df = df.join(df_clima, how="left")

rellenar cualquier hueco

In [6]:
df = df.interpolate().ffill().bfill()